- O que vamos construir: um motor simples de similaridade de termos com TF-IDF + n-grams.

- Por que isso funciona: aproximação de “fuzzy match” textual sem modelo profundo.

In [1]:
import string
import unicodedata
from typing import Any, List, Tuple, Union

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, isspmatrix_csr


## Preparar um dataset simples para a aula

In [2]:
terms_base = [
    "Antonio da Silva",
    "Antônio Silva",
    "Ana Silva",
    "Joao Pedro Souza",
    "João P. Souza",
    "Empresa Financeira",
    "Banco do Brasil",
    "Bradesco",
    "Santander",
    "Sistema Legado",
    "Fretamento",
]

queries = [
    "antonio silva",
    "joao souza",
    "banco",
    "legado",
]

## Normalização de texto: `clean()`

Objetivo didático: mostrar remoção de acento, filtragem de caracteres e padronização.

In [3]:
def clean(s: str) -> str:
    uncoded = ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in string.whitespace + string.ascii_letters + string.digits + '&'
    )
    return ' '.join(w.capitalize() for w in uncoded.split())

In [5]:
for x in ["antônio, da silva", "legado", "Brasil & Banco"]:
    print(x, "=>", clean(x))

antônio, da silva => Antonio Da Silva
legado => Legado
Brasil & Banco => Brasil & Banco


- Trade-off: você está descartando pontuação e vários símbolos (bom para nomes/termos, ruim se você precisar preservar sinais).

- Capitalizar é estética; para modelo, normalmente preferimos lower() (mas como você filtra acento, tanto faz desde que consistente).

## n-grams: `ngrams()`

Objetivo didático: mostrar por que n-gram ajuda com typos, abreviações e variações.

In [6]:
def ngrams(s: str, n: int = 3) -> List[str]:
    s = clean(s)
    ngrams_zip = zip(*[s[i:] for i in range(n)])
    return [w for w in s.split() if len(w) > n] + [''.join(ngram) for ngram in ngrams_zip]

In [7]:
print(ngrams("Antonio", 3))
print(ngrams("João P. Souza", 3))

['Antonio', 'Ant', 'nto', 'ton', 'oni', 'nio']
['Joao', 'Souza', 'Joa', 'oao', 'ao ', 'o P', ' P ', 'P S', ' So', 'Sou', 'ouz', 'uza']


- Você inclui palavras inteiras (com len(w) > n) + n-grams de caracteres.

- Isso aumenta recall (acha mais coisas), mas pode aumentar falsos positivos se o limiar for baixo.

## Chunking para performance: `divide_chunks()`

Objetivo didático: por que batch existe e como economiza memória/tempo.

In [8]:
def divide_chunks(l: List[Any], n: int, only_indexes: bool = False) -> Union[List[Any], Tuple[int, int]]: # type: ignore
    length = len(l)
    for i in range(0, length, n):
        if only_indexes:
            yield (i, min(i + n, length))
        else:
            yield l[i:i + n]

In [9]:
list(divide_chunks(list(range(10)), 4, only_indexes=True))

[(0, 4), (4, 8), (8, 10)]

## Conceito central: `TF-IDF` com analyzer customizado

Objetivo didático: mostrar que o `TfidfVectorizer` pode “tokenizar” do seu jeito.

In [10]:
vectorizer = TfidfVectorizer(analyzer=ngrams)
X = vectorizer.fit_transform(terms_base)

print("shape:", X.shape)
print("tipo:", type(X))
print("nnz (itens não-zero):", X.nnz)

shape: (11, 104)
tipo: <class 'scipy.sparse._csr.csr_matrix'>
nnz (itens não-zero): 137


- X é uma matriz esparsa (sparse): eficiente para vocabulário grande.

- analyzer=ngrams faz o vocabulário crescer muito — por isso sparse é essencial.

## Similaridade via produto escalar (aprox. do cosseno)

No seu código, a busca é query_vector.dot(self._search_vector). Isso funciona bem porque:

self._search_vector é a base transposta

o dot vira “score” entre query e cada termo da base.

In [11]:
search_vector = X.transpose().tocsr()

Q = vectorizer.transform(queries).tocsr()
scores_matrix = Q.dot(search_vector)  # (num_queries x num_base_terms)

scores_matrix.shape, type(scores_matrix)

((4, 11), scipy.sparse._csr.csr_matrix)

In [12]:
# Para ver os melhores matches de uma query específica:
q_idx = 0
row = scores_matrix[q_idx]
top_idx = np.argsort(row.toarray().ravel())[::-1][:5]

print("Query:", queries[q_idx])
for i in top_idx:
    print(f"  {i:>2} score={row[0, i]:.4f}  term='{terms_base[i]}'")

Query: antonio silva
   1 score=1.0000  term='Antônio Silva'
   0 score=0.7953  term='Antonio da Silva'
   2 score=0.4110  term='Ana Silva'
   3 score=0.0669  term='Joao Pedro Souza'
  10 score=0.0662  term='Fretamento'


## Construindo a classe TermSimilarity (versão “de aula”)

Aqui você cola praticamente seu código, mas vale didaticamente separar em partes:

`fit()`: aprende vocabulário e cria base transposta

`transform()`: calcula scores e retorna top-N por query

`_get_csr_ntop_idx_data()`: extrai top-N sem ordenar tudo (performance)

In [13]:
class TermSimilarity:
    """Modelo para detectar similaridades entre termos (TF-IDF + n-grams)."""

    def __init__(self, limit: int = 1000, minimum_score: float = 0.7, batch_size: int = 100, **tfidf_kwargs: Any):
        self._limit = limit
        self._score = minimum_score
        self._batch_size = batch_size

        kwargs = dict(analyzer=ngrams)
        kwargs.update(tfidf_kwargs)
        self._vectorizer = TfidfVectorizer(**kwargs)

        self._search_vector: csr_matrix = None
        self._terms: List[str] = None

    def _get_csr_ntop_idx_data(self, csr_row: csr_matrix) -> List[Tuple[int, float]]:
        ntop = self._limit
        lower_bound = self._score

        nnz = csr_row.getnnz()
        if nnz == 0:
            return []
        elif nnz <= ntop:
            result = zip(csr_row.indices, np.clip(csr_row.data, 0., 1.))
        else:
            arg_idx = np.argpartition(csr_row.data, -ntop)[-ntop:]
            result = zip(csr_row.indices[arg_idx], np.clip(csr_row.data[arg_idx], 0., 1.))

        return sorted(filter(lambda r: r[1] >= lower_bound, result), key=lambda x: -x[1])

    def _scipy_cosine_top(self, query_vector: csr_matrix) -> List[List[Tuple[int, float]]]:
        result_vector = query_vector.dot(self._search_vector)
        return [self._get_csr_ntop_idx_data(row) for row in result_vector]

    def fit(self, terms: List[str]) -> None:
        self._terms = terms
        self._search_vector = self._vectorizer.fit_transform(terms).transpose()
        if not isspmatrix_csr(self._search_vector):
            self._search_vector = self._search_vector.tocsr()

    def transform(self, query: List[str]) -> List[List[Tuple[int, float]]]:
        query_vector = self._vectorizer.transform(query)
        if not isspmatrix_csr(query_vector):
            query_vector = query_vector.tocsr()

        scores = []
        for start_idx, end_idx in divide_chunks(query, self._batch_size, only_indexes=True):
            scores.extend(self._scipy_cosine_top(query_vector[start_idx:end_idx, :]))
        return scores

## Teste controlado e “pretty print” do resultado

In [14]:
sim = TermSimilarity(limit=5, minimum_score=0.2, batch_size=2)
sim.fit(terms_base)

results = sim.transform(queries)

for q, hits in zip(queries, results):
    print("\nQUERY:", q)
    for idx, score in hits:
        print(f"  score={score:.4f}  term='{terms_base[idx]}'")


QUERY: antonio silva
  score=1.0000  term='Antônio Silva'
  score=0.7953  term='Antonio da Silva'
  score=0.4110  term='Ana Silva'

QUERY: joao souza
  score=0.7974  term='João P. Souza'
  score=0.7215  term='Joao Pedro Souza'

QUERY: banco
  score=0.5127  term='Banco do Brasil'

QUERY: legado
  score=0.5976  term='Sistema Legado'


- Ajustar minimum_score muda muito o comportamento (precisão vs recall).

- limit controla custo e volume de retorno.

## Exercícios rápidos (para alunos rodarem)

Markdown (célula):

Troque n=3 para n=2 e compare: melhora recall? piora falsos positivos?

Aumente a base com 2000 termos “ruins” e meça tempo/memória.

Teste com abreviações: “J P Souza”, “Sicob” (typo), etc.

Proponha um pós-processamento: remover auto-match idêntico, ou boost por palavra inteira.

## Discussão técnica final: limitações e melhorias práticas

Markdown (célula):

Limitações:

TF-IDF + n-gram não entende semântica (ex.: “carro” ~ “automóvel”).

Crescimento do vocabulário (memória).

Melhorias:

Ajustar min_df, max_df, max_features no TF-IDF.

Considerar lowercase=True e remover capitalize() (consistência).

Criar função para retornar também o termo original e talvez um “match type”.

Alternativa: embeddings + ANN (quando semântica importa).